# Your First PINN from Scratch

**Time: ~40 minutes**

We solve the ODE `u'(t) = -u(t)` with `u(0) = 1` using a neural network — no PINN library, just raw PyTorch. Every line is explained.

**Exact solution:** `u(t) = exp(-t)`

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
print("Ready.")

## Step 1: Define the Network

A simple MLP with `tanh` activation. Input: `t` (scalar). Output: `u(t)` (scalar).

In [ ]:
class SimplePINN(nn.Module):
    def __init__(self, hidden_size=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, 1),
        )

    def forward(self, t):
        return self.net(t)

model = SimplePINN()
print(f"Parameters: {sum(p.numel() for p in model.parameters())}")
print(model)

## Step 2: Define the Loss Functions

Two losses:
1. **Physics loss**: the ODE residual `u'(t) + u(t) = 0` at collocation points
2. **Initial condition loss**: `u(0) = 1`

In [ ]:
# Collocation points: where the ODE must hold
N_physics = 50
t_physics = torch.linspace(0, 2, N_physics).unsqueeze(1).requires_grad_(True)

# Initial condition point
t_ic = torch.zeros(1, 1)  # t = 0
u_ic = torch.ones(1, 1)   # u(0) = 1

def physics_loss(model):
    """ODE residual: u'(t) + u(t) = 0."""
    u = model(t_physics)
    du_dt = torch.autograd.grad(
        u, t_physics, torch.ones_like(u), create_graph=True
    )[0]
    residual = du_dt + u  # u' + u = 0
    return torch.mean(residual**2)

def ic_loss(model):
    """Initial condition: u(0) = 1."""
    u_pred = model(t_ic)
    return torch.mean((u_pred - u_ic)**2)

print(f"Physics points: {N_physics} in [0, 2]")
print(f"IC: u(0) = 1")

## Step 3: The Training Loop

This is the complete PINN training loop — no magic, no library.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
n_epochs = 5000

# Weight the IC loss higher to ensure it's satisfied
w_physics = 1.0
w_ic = 10.0

losses = {"physics": [], "ic": [], "total": []}

for epoch in range(n_epochs):
    optimizer.zero_grad()

    loss_p = physics_loss(model)
    loss_ic = ic_loss(model)
    total = w_physics * loss_p + w_ic * loss_ic

    total.backward()
    optimizer.step()

    losses["physics"].append(loss_p.item())
    losses["ic"].append(loss_ic.item())
    losses["total"].append(total.item())

    if epoch % 1000 == 0:
        print(f"Epoch {epoch:5d} | physics={loss_p.item():.4e} | ic={loss_ic.item():.4e} | total={total.item():.4e}")

print(f"\nFinal loss: {losses['total'][-1]:.4e}")

## Step 4: Visualize the Loss History

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(losses["total"], label="Total", alpha=0.8)
ax.semilogy(losses["physics"], label="Physics", alpha=0.8)
ax.semilogy(losses["ic"], label="IC", alpha=0.8)
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss (log scale)")
ax.set_title("Training Loss History")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Step 5: Compare with the Exact Solution

In [ ]:
t_test = torch.linspace(0, 2, 200).unsqueeze(1)
with torch.no_grad():
    u_pred = model(t_test).numpy()
u_exact = np.exp(-t_test.numpy())

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Solution comparison
axes[0].plot(t_test.numpy(), u_exact, 'b-', label='Exact: exp(-t)', linewidth=2)
axes[0].plot(t_test.numpy(), u_pred, 'r--', label='PINN prediction', linewidth=2)
axes[0].set_xlabel('t'); axes[0].set_ylabel('u(t)')
axes[0].set_title('Solution Comparison')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Pointwise error
error = np.abs(u_pred - u_exact)
axes[1].semilogy(t_test.numpy(), error)
axes[1].set_xlabel('t'); axes[1].set_ylabel('|error|')
axes[1].set_title(f'Pointwise Error (max = {error.max():.2e})')
axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

# Relative L2 error
rel_l2 = np.linalg.norm(u_pred - u_exact) / np.linalg.norm(u_exact)
print(f"Relative L2 error: {rel_l2:.4e}")

## Step 6: Check the Residual Everywhere

A trained PINN can self-validate: compute the PDE residual on a fine grid.

In [ ]:
t_check = torch.linspace(0, 2, 500).unsqueeze(1).requires_grad_(True)
u_check = model(t_check)
du_dt_check = torch.autograd.grad(
    u_check, t_check, torch.ones_like(u_check), create_graph=False
)[0]
residual_check = (du_dt_check + u_check).detach().numpy()

plt.figure(figsize=(8, 4))
plt.plot(t_check.detach().numpy(), residual_check)
plt.axhline(y=0, color='k', linewidth=0.5)
plt.xlabel('t'); plt.ylabel('Residual: u\' + u')
plt.title(f'PDE Residual (max |r| = {np.abs(residual_check).max():.2e})')
plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print("If this is near zero everywhere, the network truly satisfies the ODE.")

## What Just Happened?

Let's recap the complete PINN workflow:

1. **Define a neural network** that maps coordinates → solution values
2. **Compute derivatives** of the network output using `torch.autograd.grad`
3. **Build the PDE residual** from the derivatives
4. **Add IC/BC losses** to pin down the specific solution
5. **Train** with standard gradient descent (Adam)
6. **Evaluate** — the trained network IS the solution function

No mesh. No discretization scheme. No numerical stability concerns from finite differences.

The network learned `exp(-t)` just from knowing:
- The equation: `u' = -u`
- One point: `u(0) = 1`

## Exercises

1. **Change the ODE**: Try `u' = -2u` (solution: `exp(-2t)`). Just change the residual.
2. **Add more collocation points**: Try 200 instead of 50. Does accuracy improve?
3. **Try a larger domain**: Extend to `t in [0, 5]`. Where does the PINN struggle?
4. **Change the IC**: Try `u(0) = 2`. The solution should be `2*exp(-t)`.
5. **Remove the IC loss**: What happens? (The network will find *a* solution to `u' + u = 0`, but which one?)

## What's Next

**Notebook 04** tackles the same ODE three ways — pure data, pure physics, and hybrid — to show exactly when and why the physics loss matters.